In [3]:
import geopandas as gpd
from pathlib import Path
import ee
import os
import pandas as pd
import geopandas as gpd
import json
import datetime



In [4]:
import importlib
import sys

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import label_utils
importlib.reload(label_utils)

from label_utils import (
    get_s2_s1_matching_dates,
    get_s2_all_dates_with_s1,
    image_details_to_json,
    plot_dates,
    select_comparison_dates,
)

In [4]:


from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [5]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [6]:
site_fp = PROJECT_ROOT / "configs" / "sites.json"
images_fp = PROJECT_ROOT / "configs"/ "images.json"


In [8]:
ee_list_matching_dates = get_s2_s1_matching_dates(location = "RawaPening",project_root= PROJECT_ROOT, site_fp= site_fp, cloud_perc= 50)

In [15]:
print(type(ee_list_matching_dates))
print(ee_list_matching_dates["date"].unique())

<class 'pandas.core.frame.DataFrame'>
['2019-01-09' '2019-03-10' '2019-03-30' '2019-04-09' '2019-05-09'
 '2019-05-29' '2019-06-08' '2019-07-08' '2019-07-28' '2019-09-06'
 '2019-09-26' '2019-10-06' '2019-11-05' '2019-11-25' '2020-01-24'
 '2020-02-03' '2020-04-03' '2020-07-02' '2020-08-01' '2020-08-31'
 '2020-09-20' '2020-09-30' '2020-11-19' '2021-02-27' '2021-03-19'
 '2021-05-28' '2021-07-27' '2021-08-26' '2021-09-25' '2021-10-25'
 '2021-12-24' '2022-03-14' '2023-03-09' '2023-06-17' '2023-08-16'
 '2023-09-05' '2023-10-15' '2023-12-14' '2024-02-12' '2024-04-12'
 '2024-05-02' '2024-06-11' '2024-07-01' '2024-08-10' '2024-08-30'
 '2024-10-29' '2025-02-06' '2025-04-07' '2025-05-07' '2025-05-19'
 '2025-06-06' '2025-06-26' '2025-07-18' '2025-08-25' '2025-09-04'
 '2025-10-16' '2025-12-23']


In [11]:
# Import vembanad matching dates 
dates_fp = PROJECT_ROOT /"outputs/RawaPening_matching_dates.csv"
matching_dates_df = pd.read_csv(dates_fp)

In [12]:
fig = plot_dates(location = "RawaPening", project_root= PROJECT_ROOT, start_year= 2022, end_year=2025)
fig.show()

In [28]:
# Next want a function that takes a matching dates Df and writes the info into images.json

# How to chose the comparison images???

comparison_df = select_comparison_dates(matching_dates= matching_dates_df, months_back= 3, max_cloud_perc= 60)

In [51]:
comparison_df.head()

,Unnamed: 0,S1_img_id,S1_time,S2_img_id,S2_time,cloud_perc,date,location,time_diff,season,...,comparison_1_date,comparison_1_s2_img_id,comparison_1_s1_img_id,comparison_1_cloud_perc,comparison_1_age_days,comparison_2_date,comparison_2_s2_img_id,comparison_2_s1_img_id,comparison_2_cloud_perc,comparison_2_age_days
0,1,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,2019-01-07 00:39:13+00:00,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,2019-01-07 17:18:20.897000+00:00,0.012322,2019-01-07,Valsequillo,0 days 16:39:07.897000,winter,...,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
1,3,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,2019-01-22 12:26:09+00:00,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,2019-01-22 17:18:26.325000+00:00,0.000185,2019-01-22,Valsequillo,0 days 04:52:17.325000,winter,...,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,15.0,NaT,NaN,NaN,NaN,NaN
2,4,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190206T00...,2019-02-06 00:40:10+00:00,COPERNICUS/S2_SR_HARMONIZED/20190206T170501_20...,2019-02-06 17:18:37.029000+00:00,0.000266,2019-02-06,Valsequillo,0 days 16:38:27.029000,winter,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,15.0,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,30.0
3,6,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190221T12...,2019-02-21 12:25:14+00:00,COPERNICUS/S2_SR_HARMONIZED/20190221T170329_20...,2019-02-21 17:18:39.407000+00:00,0.006086,2019-02-21,Valsequillo,0 days 04:53:25.407000,winter,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,30.0,2019-02-06,COPERNICUS/S2_SR_HARMONIZED/20190206T170501_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190206T00...,0.000266,15.0
4,8,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190308T00...,2019-03-08 00:39:11+00:00,COPERNICUS/S2_SR_HARMONIZED/20190308T170131_20...,2019-03-08 17:18:34.608000+00:00,0.000448,2019-03-08,Valsequillo,0 days 16:39:23.608000,spring,...,2019-01-22,COPERNICUS/S2_SR_HARMONIZED/20190122T170619_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20190122T12...,0.000185,45.0,2019-01-07,COPERNICUS/S2_SR_HARMONIZED/20190107T170701_20...,COPERNICUS/S1_GRD/S1B_IW_GRDH_1SDV_20190107T00...,0.012322,60.0


In [55]:
# Write image details to images.json from camparison_df that is the output from select_comparison_dates()

image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)

NameError: name 'image_details_to_json' is not defined

In [10]:
# Loop through rest of locations:

with open(site_fp) as f:
    sites = json.load(f)

location_list = list(sites.get("sites", {}).keys())

for location in location_list:
    matching_dates = get_s2_s1_matching_dates(location, project_root= PROJECT_ROOT, site_fp= site_fp, cloud_perc= 40)

    comparison_df = select_comparison_dates(matching_dates= matching_dates, months_back= 4, n_comparisons=2, max_cloud_perc=40, min_gap_days=7)

    fig= plot_dates(location=location, start_year= 2021, end_year=2025, project_root= PROJECT_ROOT)
    fig.show()
    image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)
    print(f"Ran for {location}")

Ran for Vembanad


Ran for Winam


Ran for Inle


Ran for Hartbeespoort


Ran for Mula


Ran for RawaPening


Ran for Rodman


Ran for Valsequillo


In [21]:
#### To get more images for locations with few matching overpasses run an different function that does a left join of s1 onto s2 instead of the inner join
# so can see more image choices even if no match. Also useful for extra comparison images. 

location_subset = ["RawaPening", "Rodman", "Mula", "Vembanad"]

for location in location_subset:
    dates = get_s2_all_dates_with_s1(site_fp=site_fp, location= location, project_root= PROJECT_ROOT, cloud_perc= 30)

    comparison_df = select_comparison_dates(matching_dates= dates, months_back= 6, n_comparisons=2, max_cloud_perc=10, min_gap_days= 14)

    image_dict = image_details_to_json(images_fp= images_fp, comparison_df= comparison_df)
    print(f"Ran for {location}")

    

Ran for RawaPening
Ran for Rodman
Ran for Mula
Ran for Vembanad


### Create table of number of labels


In [5]:
samples = PROJECT_ROOT / "outputs" / "sample_points_clean.csv"

cleaned_samples = pd.read_csv(samples)

cleaned_samples.head()

,AWEIp95,B11,B12,B2,B3,B4,B5,B6,B7,B8,...,NDBI,NDAVI,RR1,EVI,SAVI,DVI,OSAVI,AWEIsh,SABI,BSI
0,0.2161,0.0294,0.0174,0.0431,0.0584,0.0327,0.0444,0.0776,0.0759,0.1560,...,-0.682848,0.567052,0.151751,0.000031,0.268549,0.0976,0.353599,-0.093350,1.214778,-0.524502
1,0.2191,0.0256,0.0133,0.0406,0.0472,0.0301,0.0367,0.0564,0.0553,0.0836,...,-0.531136,0.346216,0.098802,0.000013,0.130764,0.0364,0.195469,-0.008525,0.609339,-0.380767
2,0.2214,0.0173,0.0112,0.0430,0.0506,0.0323,0.0382,0.0479,0.0477,0.0516,...,-0.497823,0.090909,0.083688,0.000005,0.049580,0.0010,0.079131,0.063350,0.206197,-0.312067
3,0.2239,0.0254,0.0119,0.0431,0.0478,0.0301,0.0429,0.0683,0.0684,0.0363,...,-0.176661,-0.085642,0.175342,0.000002,0.016419,-0.0115,0.027385,0.067075,0.068207,-0.177168
4,0.2148,0.0144,0.0093,0.0464,0.0506,0.0330,0.0372,0.0430,0.0415,0.0490,...,-0.545741,0.027254,0.059829,0.000004,0.041237,-0.0016,0.066116,0.075475,0.164948,-0.336134


In [ ]:
cleaned_samples["obs_date"] = pd.to_datetime(cleaned_samples["obs_date"],format="%Y-%m-%d")
cleaned_samples = cleaned_samples[["class_label", "lc", "class_int", "location", "obs_date"]]

grouped_samples = cleaned_samples.drop(columns =["class_int"]).groupby(["location", "lc", "class_label"]).count()

grouped_samples

In [38]:

by_date_1 = cleaned_samples.copy()

by_date_1["year"] = by_date_1["obs_date"].dt.year

by_label_wide = by_date_1.pivot_table(
    index=["location"],
    columns=["lc", "class_label"],
    values="obs_date",
    aggfunc="count",   # counts non-null lc values
    fill_value=0,
    observed=False,    # do / don't keep all season categories even if some are unused for a given group
).reset_index()

by_label_wide[(0,"negative_total")] = by_label_wide[(0,"LEV")] + by_label_wide[(0,"open_water")] +by_label_wide[(0,"surface_algae")] 

by_label_wide["total"] = by_label_wide[(0,"LEV")] + by_label_wide[(0,"open_water")] +by_label_wide[(0,"surface_algae")] + by_label_wide[(1, "floating_plants")]


In [40]:
by_label_wide.head(20)

label_table_fp = PROJECT_ROOT / "outputs/tables" / "label_count_table.csv"
by_label_wide.to_csv(label_table_fp)

In [42]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

# Can make seson an ordered categorical so it list in season order rather than alphabetical
from pandas.api.types import CategoricalDtype


by_date = cleaned_samples.copy()
by_date["year"] = by_date["obs_date"].dt.year
by_date["month"] = by_date["obs_date"].dt.month




season_order = CategoricalDtype(
    ["Winter", "Spring", "Summer", "Autumn"],
    ordered=True,
)

by_date["season"] = (
    by_date["obs_date"]
    .dt.month
    .map(season_map)
    .astype(season_order)
)

# by_date = by_date.drop(columns=["class_int"]).groupby(["location", "year", "season"]).count()
# by_date = by_date.loc[by_date["obs_date"]!= 0]

by_date_wide = by_date.pivot_table(
    index=["location", "year"],
    columns="season",
    values="lc",
    aggfunc="count",   # counts non-null lc values
    fill_value=0,
    observed=True,    # do / don't keep all season categories even if some are unused for a given group
).reset_index()

by_date_wide["total"] = by_date_wide["Winter"] + by_date_wide["Spring"]+ by_date_wide["Summer"]+ by_date_wide["Autumn"]

In [43]:
by_date_wide.head(20)

date_table_fp = PROJECT_ROOT / "outputs/tables" / "date_season_count_table.csv"
by_date_wide.to_csv(date_table_fp)